# Exploratory Data Analysis -- Home Credit Default Risk

Covers dataset summary, data quality, feature categorization, and the 5 key financial insights. Mirrors `notebooks/eda.py` and the Streamlit app's EDA tab.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import matplotlib.pyplot as plt

from src.data.loader import load_raw_data
from src.data.preprocessor import engineer_features

raw_df, msg = load_raw_data()
print(msg)
df = engineer_features(raw_df)
df.shape

## Dataset Summary & Data Quality

In [ ]:
print(f"Applicants: {len(df):,}")
print(f"Default rate: {df['TARGET'].mean()*100:.2f}%")
print(f"Average missingness: {df.isna().mean().mean()*100:.1f}%")
df.isna().mean().sort_values(ascending=False).head(8)

**Data quality notes:**
- `DAYS_EMPLOYED == 365243` is Home Credit's documented anomaly code for pensioners/unemployed applicants; decoded into `EMPLOYED_ANOMALY_FLAG` rather than treated as a real tenure value.
- `EXT_SOURCE_1/2/3` carry meaningful missingness (bureau score not always available), captured via `EXT_SOURCE_MISSING_COUNT` rather than naively imputed.

## Feature Categorization

| Category | Examples |
|---|---|
| Demographics | `CODE_GENDER`, `CNT_CHILDREN`, `NAME_FAMILY_STATUS`, `AGE_YEARS` |
| Financials | `AMT_INCOME_TOTAL`, `AMT_CREDIT`, `AMT_ANNUITY`, `AMT_GOODS_PRICE` |
| Credit history / bureau | `EXT_SOURCE_1/2/3`, `AMT_REQ_CREDIT_BUREAU_QRT` |
| Employment & stability | `NAME_INCOME_TYPE`, `EMPLOYED_YEARS`, `OCCUPATION_TYPE` |
| Housing | `NAME_HOUSING_TYPE`, `FLAG_OWN_REALTY`, `FLAG_OWN_CAR` |

## Insight 1: External Score Non-Linearity

In [ ]:
bins = pd.qcut(df['EXT_SOURCE_MEAN'].fillna(df['EXT_SOURCE_MEAN'].median()), 8, duplicates='drop')
df.groupby(bins, observed=True)['TARGET'].mean().plot(kind='bar', figsize=(8,4), title='Default Rate by External Bureau Score Bucket')
plt.ylabel('Default Rate'); plt.tight_layout(); plt.show()

Default rate drops sharply -- not linearly -- as the mean external bureau score rises, steepest in the bottom two buckets.

## Insight 2: The Debt-Trap Ratio

In [ ]:
dti_bins = pd.cut(df['ANNUITY_TO_INCOME'].clip(upper=1.0), bins=[0, 0.15, 0.25, 0.35, 0.5, 1.0])
df.groupby(dti_bins, observed=True)['TARGET'].mean().plot(kind='bar', figsize=(8,4), title='Default Rate by Annuity-to-Income Bucket', color='orange')
plt.ylabel('Default Rate'); plt.tight_layout(); plt.show()

Default risk climbs once the annuity-to-income ratio crosses ~35% -- the threshold used in the Tier 1 hard knockout rule.

## Insight 3: Employment & Age Stability

In [ ]:
age_bins = pd.cut(df['AGE_YEARS'], bins=[20, 30, 40, 50, 60, 70])
df.groupby(age_bins, observed=True)['TARGET'].mean().plot(kind='bar', figsize=(8,4), title='Default Rate by Age Bracket', color='green')
plt.ylabel('Default Rate'); plt.tight_layout(); plt.show()

Younger applicants (20-30) show a materially higher default rate than older, more employment-stable brackets.

## Insight 4: Over-Financing Risk

In [ ]:
over_financed = df['CREDIT_TO_GOODS_RATIO'] > 1.0
df.assign(OVER_FINANCED=over_financed).groupby('OVER_FINANCED', observed=True)['TARGET'].mean().plot(kind='bar', figsize=(6,4), title='Default Rate: Over-Financed vs. Not', color='red')
plt.ylabel('Default Rate'); plt.tight_layout(); plt.show()

Applicants whose requested credit exceeds the goods price (under-collateralized lending) default at a visibly higher rate.

## Insight 5: Credit Bureau Inquiry Bursts

In [ ]:
df.groupby('AMT_REQ_CREDIT_BUREAU_QRT', observed=True)['TARGET'].mean().plot(kind='bar', figsize=(8,4), title='Default Rate by Bureau Inquiries (last quarter)', color='purple')
plt.ylabel('Default Rate'); plt.tight_layout(); plt.show()

A burst of recent bureau inquiries -- a signal of credit-seeking distress -- tracks with higher default rates.